# Colab Training Notebook

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q torchmetrics mlflow tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 25.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 81.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 81.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [3]:
import torch
import shutil, os, glob
import mlflow
import pandas as pd
import sys
import argparse
from dotenv import load_dotenv
from pathlib import Path

In [4]:
load_dotenv()

True

In [5]:
# Check GPU
if torch.cuda.is_available():
    print(f"CUDA available: {torch.cuda.is_available()}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU")

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [6]:
# Set Drive path
DRIVE_PATH = Path(os.environ["DRIVE_PATH"])
os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/data/splits', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/data/labeled', exist_ok=True)
print("Drive path ready")

Drive path ready


In [7]:
if str(DRIVE_PATH) not in sys.path:
    sys.path.append(str(DRIVE_PATH))

from src.train import main as train_main

In [8]:
# Set MLflow tracking URI to local SQLite database
mlflow.set_tracking_uri(f"sqlite:///{DRIVE_PATH}/mlflow.db")
mlflow.set_experiment("visiomark-image-model")
print("MLflow ready")

MLflow ready


In [9]:
# Check data splits
splits = {}
for split in ['train', 'val', 'test']:
    path = f"{DRIVE_PATH}/data/splits/{split}.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        splits[split] = df
        print(f"{split}: {len(df)} samples")
        print(f"content_type distribution: {df['content_type_label'].value_counts().to_dict()}")
        print(f"mood distribution:         {df['mood_label'].value_counts().to_dict()}")
        print()
    else:
        print(f"{split}.csv not found")

train: 1131 samples
content_type distribution: {1: 609, 2: 355, 0: 167}
mood distribution:         {1: 423, 2: 352, 0: 224, 3: 132}

val: 243 samples
content_type distribution: {1: 131, 2: 76, 0: 36}
mood distribution:         {1: 91, 2: 76, 0: 47, 3: 29}

test: 243 samples
content_type distribution: {1: 131, 2: 76, 0: 36}
mood distribution:         {1: 90, 2: 76, 0: 48, 3: 29}



### Phase 1

In [10]:
print("Starting Phase 1.")
print("=" * 60)

args_phase1 = argparse.Namespace(
    phase=1,
    epochs=15,
    batch_size=16,
    lr=1e-3,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    train_csv=f"{DRIVE_PATH}/data/splits/train.csv",
    val_csv=f"{DRIVE_PATH}/data/splits/val.csv",
    checkpoint=None,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir=f"{DRIVE_PATH}/checkpoints",
)

checkpoints_path = train_main(args_phase1)
print("Phase 1 checkpoints saved to: /checkpoints ")

Starting Phase 1.
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 152MB/s]


Epoch 01/15 | loss=2.3202 | F1_content=0.6202 | F1_mood=0.4748 | Avg_F1=0.5475


2026/06/07 15:28:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 15:28:58 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/07 15:28:58 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/07 15:29:09 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

    Checkpoint saved (avg_F1=0.5475)


...

    Checkpoint saved (avg_F1=0.6217)


2026/06/07 15:34:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 14/15 | loss=1.6124 | F1_content=0.6659 | F1_mood=0.5797 | Avg_F1=0.6228


2026/06/07 15:34:54 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/07 15:34:54 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/07 15:35:01 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the loca

    Checkpoint saved (avg_F1=0.6228)
Epoch 15/15 | loss=1.6014 | F1_content=0.6336 | F1_mood=0.5660 | Avg_F1=0.5998

Phase 1 done. Best avg F1 = 0.6228
Phase 1 checkpoints saved to: /checkpoints 


### Phase 2

In [11]:
print("Starting Phase 2.")
print("=" * 60)

phase1_checkpoints = sorted(glob.glob(f'{DRIVE_PATH}/checkpoints/best_phase1*.pt'))

if not phase1_checkpoints:
    raise FileNotFoundError("No Phase 1 checkpoint found.")

best_phase1 = phase1_checkpoints[-1]

args_phase2 = argparse.Namespace(
    phase=2,
    epochs=50,
    batch_size=16,
    lr=1e-5,
    img_dir=f"{DRIVE_PATH}/data/raw/",
    train_csv=f"{DRIVE_PATH}/data/splits/train.csv",
    val_csv=f"{DRIVE_PATH}/data/splits/val.csv",
    checkpoint=best_phase1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    save_dir=f"{DRIVE_PATH}/checkpoints",
)

final_checkpoints_path = train_main(args_phase2)
print("Phase 2 checkpoints saved to: /checkpoints ")

Starting Phase 2.
  Loading Phase 1 checkpoint
  Unfroze last 3 encoder blocks
Epoch 01/50 | loss=1.6027 | F1_content=0.6632 | F1_mood=0.5829 | Avg_F1=0.6231


2026/06/07 15:35:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/07 15:35:54 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/07 15:35:54 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/07 15:36:00 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

    Checkpoint saved (avg_F1=0.6231)
Epoch 02/50 | loss=1.5823 | F1_content=0.6464 | F1_mood=0.5815 | Avg_F1=0.6139
Epoch 03/50 | loss=1.5343 | F1_content=0.6451 | F1_mood=0.5827 | Avg_F1=0.6139
Epoch 04/50 | loss=1.4792 | F1_content=0.6557 | F1_mood=0.5855 | Avg_F1=0.6206
Epoch 05/50 | loss=1.5062 | F1_content=0.6491 | F1_mood=0.5937 | Avg_F1=0.6214
Epoch 06/50 | loss=1.4708 | F1_content=0.6694 | F1_mood=0.5755 | Avg_F1=0.6225


2026/06/07 15:38:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 07/50 | loss=1.4540 | F1_content=0.6767 | F1_mood=0.5867 | Avg_F1=0.6317


...

    Checkpoint saved (avg_F1=0.6959)
Epoch 34/50 | loss=1.0473 | F1_content=0.7010 | F1_mood=0.6557 | Avg_F1=0.6783
Epoch 35/50 | loss=1.0426 | F1_content=0.7084 | F1_mood=0.6475 | Avg_F1=0.6779
Epoch 36/50 | loss=1.0282 | F1_content=0.7035 | F1_mood=0.6416 | Avg_F1=0.6725
Epoch 37/50 | loss=1.0167 | F1_content=0.7207 | F1_mood=0.6213 | Avg_F1=0.6710


2026/06/07 15:53:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 38/50 | loss=1.0592 | F1_content=0.7342 | F1_mood=0.6593 | Avg_F1=0.6967


2026/06/07 15:53:32 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/07 15:53:32 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/07 15:53:38 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the loca

    Checkpoint saved (avg_F1=0.6967)
Epoch 39/50 | loss=1.0389 | F1_content=0.6989 | F1_mood=0.6443 | Avg_F1=0.6716
Epoch 40/50 | loss=1.0333 | F1_content=0.7163 | F1_mood=0.6707 | Avg_F1=0.6935
Epoch 41/50 | loss=1.0147 | F1_content=0.7265 | F1_mood=0.6631 | Avg_F1=0.6948
Epoch 42/50 | loss=1.0410 | F1_content=0.7154 | F1_mood=0.6384 | Avg_F1=0.6769
Epoch 43/50 | loss=1.0561 | F1_content=0.7165 | F1_mood=0.6666 | Avg_F1=0.6915
Epoch 44/50 | loss=1.0186 | F1_content=0.7154 | F1_mood=0.6609 | Avg_F1=0.6881
Epoch 45/50 | loss=0.9881 | F1_content=0.7138 | F1_mood=0.6510 | Avg_F1=0.6824
Epoch 46/50 | loss=1.0087 | F1_content=0.7159 | F1_mood=0.6566 | Avg_F1=0.6863
Epoch 47/50 | loss=0.9833 | F1_content=0.7200 | F1_mood=0.6605 | Avg_F1=0.6902
Epoch 48/50 | loss=0.9918 | F1_content=0.7042 | F1_mood=0.6483 | Avg_F1=0.6763
Epoch 49/50 | loss=1.0294 | F1_content=0.6952 | F1_mood=0.6589 | Avg_F1=0.6770
Epoch 50/50 | loss=1.0126 | F1_content=0.7066 | F1_mood=0.6463 | Avg_F1=0.6764

Phase 2 done. 

In [12]:
# Copy MLflow DB if using local mode
db_source = f"{DRIVE_PATH}/mlflow.db"
db_backup = f"{DRIVE_PATH}/mlflow_backup.db"

if os.path.exists(db_source):
    shutil.copy2(db_source, db_backup)
    print("MLflow database backed up saved.")
else:
    print("MLflow DB not found.")

print("\nALL TRAINING COMPLETE!")

MLflow database backed up saved.

ALL TRAINING COMPLETE!
